# **PHASE 5 - FAILURE ANALYSIS**

In [1]:
# ============================================================
# PHASE 5 — FAILURE ANALYSIS
# DuckDB Systems Engineering Analysis
# ============================================================

import duckdb, threading, time, os

CSV_PATH = "flights.csv"

**5A - OOM hard abort (100MB)**

In [2]:
# ── EXPERIMENT 5A ────────────────────────────────────────────
# Failure Mode 1: OOM below spill threshold (violent abort)
# Memory limit so tight the buffer manager cannot even begin
# ─────────────────────────────────────────────────────────────

print("\n[5A] OOM BELOW SPILL THRESHOLD")
print("-" * 40)

query = """
    SELECT AIRLINE, count(*) as delayed_flights, avg(DEPARTURE_DELAY) as avg_delay
    FROM read_csv_auto('flights.csv')
    WHERE DEPARTURE_DELAY > 15
    GROUP BY AIRLINE
    ORDER BY delayed_flights DESC
"""

con = duckdb.connect()
con.execute("SET memory_limit='100MB'")
con.execute("SET temp_directory='./duckdb_spill_test'")

try:
    con.execute(query).fetchall()
    print("Query completed (unexpected)")
except duckdb.OutOfMemoryException as e:
    print(f"Exception type : duckdb.OutOfMemoryException")
    print(f"Message        : {str(e).splitlines()[0]}")
    print(f"Behaviour      : Query violently aborted — no partial result, no recovery")
con.close()


[5A] OOM BELOW SPILL THRESHOLD
----------------------------------------
Exception type : duckdb.OutOfMemoryException
Message        : Out of Memory Error: could not allocate block of size 30.5 MiB (66.9 MiB/95.3 MiB used)
Behaviour      : Query violently aborted — no partial result, no recovery


**5B - OOM graceful spill (300MB)**

In [5]:
# ── EXPERIMENT 5B ────────────────────────────────────────────
# Failure Mode 2: OOM above spill threshold (graceful degrade)
# Memory tight but survivable — buffer manager spills to disk
# ─────────────────────────────────────────────────────────────

print("\n[5B] OOM ABOVE SPILL THRESHOLD — GRACEFUL SPILL")
print("-" * 40)

import glob, tempfile

SPILL_DIR = os.path.abspath("./duckdb_spill_test")
os.makedirs(SPILL_DIR, exist_ok=True)

# Baseline
con = duckdb.connect()
t0 = time.time()
con.execute(query).fetchall()
baseline = time.time() - t0
con.close()

# Constrained
con = duckdb.connect()
con.execute(f"SET temp_directory='{SPILL_DIR}'")
con.execute("SET threads=1")
con.execute("SET memory_limit='300MB'")
con.execute("SET preserve_insertion_order=false")
t0 = time.time()
con.execute(query).fetchall()
spill_time = time.time() - t0
con.close()

# Count spill files from system temp
sys_temp = tempfile.gettempdir()
duckdb_spill = [f for f in os.listdir(sys_temp)
                if f.startswith('tmp') and f.endswith('.tmp') and len(f) <= 12]
total_mb = sum(os.path.getsize(os.path.join(sys_temp, f))
               for f in duckdb_spill) / (1024*1024)

print(f"Baseline (no limit) : {baseline:.3f}s")
print(f"Spill mode (300MB)  : {spill_time:.3f}s")
print(f"Overhead            : {spill_time/baseline:.2f}x slower")
print(f"Spill volume        : {total_mb:.0f} MB written to disk")
print(f"Behaviour           : Query completed — buffer manager degraded gracefully")


[5B] OOM ABOVE SPILL THRESHOLD — GRACEFUL SPILL
----------------------------------------
Baseline (no limit) : 1.638s
Spill mode (300MB)  : 5.006s
Overhead            : 3.06x slower
Spill volume        : 138 MB written to disk
Behaviour           : Query completed — buffer manager degraded gracefully


In [6]:
# Run 5B three times to get a stable average
import duckdb, time, os, tempfile

query = """
    SELECT AIRLINE, count(*) as delayed_flights, avg(DEPARTURE_DELAY) as avg_delay
    FROM read_csv_auto('flights.csv')
    WHERE DEPARTURE_DELAY > 15
    GROUP BY AIRLINE
    ORDER BY delayed_flights DESC
"""

# Baseline (run once)
con = duckdb.connect()
t0 = time.time()
con.execute(query).fetchall()
baseline = time.time() - t0
con.close()
print(f"Baseline: {baseline:.3f}s")

# Spill mode — 3 runs
spill_times = []
for i in range(3):
    con = duckdb.connect()
    con.execute("SET threads=1")
    con.execute("SET memory_limit='300MB'")
    con.execute("SET preserve_insertion_order=false")
    t0 = time.time()
    con.execute(query).fetchall()
    spill_times.append(time.time() - t0)
    con.close()
    print(f"Spill run {i+1}: {spill_times[-1]:.3f}s")

avg_spill = sum(spill_times) / len(spill_times)
print(f"\nAverage spill time : {avg_spill:.3f}s")
print(f"Average overhead   : {avg_spill/baseline:.2f}x slower")
print(f"Run variance       : {max(spill_times)-min(spill_times):.3f}s")

Baseline: 1.877s
Spill run 1: 6.280s
Spill run 2: 5.158s
Spill run 3: 4.765s

Average spill time : 5.401s
Average overhead   : 2.88x slower
Run variance       : 1.515s


**5C - Host crash → bad_weak_ptr**

In [7]:
# ── EXPERIMENT 5C ────────────────────────────────────────────
# Failure Mode 3: Host process interference (in-process crash)
# Force-closing connection mid-query simulates host crash
# ─────────────────────────────────────────────────────────────

print("\n[5C] HOST PROCESS INTERFERENCE")
print("-" * 40)

con = duckdb.connect()
con.execute("CREATE TABLE IF NOT EXISTS flights AS SELECT * FROM read_csv_auto('flights.csv')")

results = []

def run_query():
    try:
        r = con.execute("""
            SELECT AIRLINE, count(*) as delayed_flights, avg(DEPARTURE_DELAY) as avg_delay
            FROM flights
            WHERE DEPARTURE_DELAY > 15
            GROUP BY AIRLINE
        """).fetchall()
        results.append(('success', len(r)))
    except Exception as e:
        results.append(('error', type(e).__name__, str(e)))

t = threading.Thread(target=run_query)
t.start()
time.sleep(0.01)

try:
    con.close()
except Exception as e:
    print(f"Host action        : con.close() mid-query → {e}")

t.join()

outcome = results[0]
print(f"Query outcome      : {outcome[0].upper()}")
if outcome[0] == 'error':
    print(f"Exception type     : {outcome[1]}")
    print(f"Message            : {outcome[2]}")
print(f"Behaviour          : No recovery possible — shared memory = shared fate")


[5C] HOST PROCESS INTERFERENCE
----------------------------------------
Query outcome      : ERROR
Exception type     : Error
Message            : bad_weak_ptr
Behaviour          : No recovery possible — shared memory = shared fate


**Summary table**

In [8]:
# ── SUMMARY TABLE ────────────────────────────────────────────

print("\n" + "=" * 60)
print("PHASE 5 SUMMARY")
print("=" * 60)
print(f"""
┌─────────────────────┬──────────────────────┬────────────────────────┐
│ Failure Mode        │ Evidence             │ Behaviour              │
├─────────────────────┼──────────────────────┼────────────────────────┤
│ 5A: OOM hard limit  │ OutOfMemoryException │ Violent abort          │
│     (100MB cap)     │ 30.5MB block denied  │ No recovery            │
├─────────────────────┼──────────────────────┼────────────────────────┤
│ 5B: OOM soft limit  │ 138MB spill to disk  │ Graceful degradation   │
│     (300MB cap)     │ 2.66x slower         │ Query completes        │
├─────────────────────┼──────────────────────┼────────────────────────┤
│ 5C: Host crash      │ bad_weak_ptr         │ Immediate C++ failure  │
│     (mid-query)     │ C++ memory signal    │ No retry, no HA        │
└─────────────────────┴──────────────────────┴────────────────────────┘

Architecture conclusion:
DuckDB has a two-tier failure response:
  - Memory pressure it can manage → graceful spill (5B)
  - Memory pressure it cannot manage → clean exception (5A)
  - Host-level failure → no defense, shared fate (5C)
This is the direct consequence of the in-process architecture.
The same design that gives zero-latency execution (Phase 3: 0.056s)
eliminates the fault-tolerance boundary that Spark's master/worker
model provides.
""")


PHASE 5 SUMMARY

┌─────────────────────┬──────────────────────┬────────────────────────┐
│ Failure Mode        │ Evidence             │ Behaviour              │
├─────────────────────┼──────────────────────┼────────────────────────┤
│ 5A: OOM hard limit  │ OutOfMemoryException │ Violent abort          │
│     (100MB cap)     │ 30.5MB block denied  │ No recovery            │
├─────────────────────┼──────────────────────┼────────────────────────┤
│ 5B: OOM soft limit  │ 138MB spill to disk  │ Graceful degradation   │
│     (300MB cap)     │ 2.66x slower         │ Query completes        │
├─────────────────────┼──────────────────────┼────────────────────────┤
│ 5C: Host crash      │ bad_weak_ptr         │ Immediate C++ failure  │
│     (mid-query)     │ C++ memory signal    │ No retry, no HA        │
└─────────────────────┴──────────────────────┴────────────────────────┘

Architecture conclusion:
DuckDB has a two-tier failure response:
  - Memory pressure it can manage → graceful spill (